In [1]:
import pandas as pd
import requests
import os

In [2]:
# url for all calls
base_url = "https://fantasy.premierleague.com/api/"
# folder where data will be saved
folder = r"B:\Python_Projects\PowerBI\FPL_PBI\FPL-Analysis\25-26 gws upload"
output = os.path.join(folder, "players.csv")

In [3]:
# call api
response = requests.get(base_url + "bootstrap-static/")

In [4]:
# convert to json
data = response.json()

In [5]:
print(type(data))
print(type(response))
print(response.status_code)

<class 'dict'>
<class 'requests.models.Response'>
200


In [6]:
# displaying available fields
print(data["elements"][0].keys())

dict_keys(['can_transact', 'can_select', 'chance_of_playing_next_round', 'chance_of_playing_this_round', 'code', 'cost_change_event', 'cost_change_event_fall', 'cost_change_start', 'cost_change_start_fall', 'price_change_percent', 'dreamteam_count', 'element_type', 'ep_next', 'ep_this', 'event_points', 'first_name', 'form', 'id', 'in_dreamteam', 'news', 'news_added', 'now_cost', 'photo', 'points_per_game', 'removed', 'second_name', 'selected_by_percent', 'special', 'squad_number', 'status', 'team', 'team_code', 'total_points', 'transfers_in', 'transfers_in_event', 'transfers_out', 'transfers_out_event', 'value_form', 'value_season', 'web_name', 'known_name', 'region', 'team_join_date', 'birth_date', 'has_temporary_code', 'opta_code', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'influence', 'creativity', 'threat', 'ict_index', 'clearances_blocks_intercept

In [7]:
# keeping necessary fields
# NOTE: keeping original API column names (code, element_type) so Power BI does not need to change
df = pd.DataFrame(data["elements"])
df = df[["id", "code", "first_name", "second_name", "web_name", "photo", "element_type"]]

In [8]:
df.head()

,id,code,first_name,second_name,web_name,photo,element_type
0,1,154561,David,Raya Martín,Raya,154561.jpg,1
1,2,109745,Kepa,Arrizabalaga Revuelta,Arrizabalaga,109745.jpg,1
2,3,463748,Karl,Hein,Hein,463748.jpg,1
3,4,551221,Tommy,Setford,Setford,551221.jpg,1
4,5,226597,Gabriel,dos Santos Magalhães,Gabriel,226597.jpg,2


In [9]:
# replace 'jpg' with 'png' as images don't work with jpg
df['photo'] = df['photo'].str.replace('jpg', 'png', regex=False)

In [10]:
# add prefix to photo
df['photo'] = "https://resources.premierleague.com/premierleague25/photos/players/110x140/" + df['photo']
df.head(1)

,id,code,first_name,second_name,web_name,photo,element_type
0,1,154561,David,Raya Martín,Raya,https://resources.premierleague.com/premierlea...,1


In [11]:
df.iloc[0]['photo']

'https://resources.premierleague.com/premierleague25/photos/players/110x140/154561.png'

In [12]:
# making sure there are no blank web names
(df['web_name'] == '').sum()

np.int64(0)

In [13]:
# check for no duplicates
df['id'].duplicated().sum()

np.int64(0)

In [14]:
# Save CSV with original column names (code, element_type) — no renaming
# This keeps Power BI working without any changes to the .pbix file
df.to_csv(output, index=False)
print(f"Saved {len(df)} players to {output}")
print(df.dtypes)

Saved 829 players to B:\Python_Projects\PowerBI\FPL_PBI\FPL-Analysis\25-26 gws upload\players.csv
id               int64
code             int64
first_name      object
second_name     object
web_name        object
photo           object
element_type     int64
dtype: object


In [15]:
# Quick schema check — verify all 3 CSVs look correct
import pandas as pd
from glob import glob
import os

# 1. Players file
players = pd.read_csv(r"B:\Python_Projects\PowerBI\FPL_PBI\FPL-Analysis\25-26 gws upload\players.csv")
print("=== PLAYERS ===")
print(players.dtypes)
print(players.head(2))

# 2. Player scores file
scores = pd.read_csv(r"B:\Python_Projects\PowerBI\FPL_PBI\FPL-Analysis\25-26 gws upload\player_scores.csv")
print("\n=== PLAYER SCORES ===")
print(scores.dtypes)
print(scores.head(2))

# 3. One GW file
folder = r"B:\Python_Projects\PowerBI\FPL_PBI\FPL-Analysis\25-26 gws upload"
gw_files = [f for f in glob(os.path.join(folder, "*.csv")) if "players" not in f and "player_scores" not in f]
gw = pd.read_csv(gw_files[0])
print("\n=== GW FILE ===")
print(gw.dtypes)
print(gw.head(2))

=== PLAYERS ===
id               int64
code             int64
first_name      object
second_name     object
web_name        object
photo           object
element_type     int64
dtype: object
   id    code first_name            second_name      web_name  \
0   1  154561      David            Raya Martín          Raya   
1   2  109745       Kepa  Arrizabalaga Revuelta  Arrizabalaga   

                                               photo  element_type  
0  https://resources.premierleague.com/premierlea...             1  
1  https://resources.premierleague.com/premierlea...             1  


FileNotFoundError: [Errno 2] No such file or directory: 'B:\\Python_Projects\\PowerBI\\FPL_PBI\\FPL-Analysis\\25-26 gws upload\\player_scores.csv'